# 7. AIND Ephys Results Collector

Run [aind-ephys-results-collector](https://github.com/AllenNeuralDynamics/aind-ephys-results-collector/tree/main/code) on the artifacts produced by notebooks 1-6.

The collector aggregates everything (`preprocessed_*.json`, `spikesorted_*`, `postprocessed_*.zarr`, `qc_*.npy`, `unit_classifier_*.csv`, `visualization_*`, plus all `data_process_*.json` files) into a single results layout (`results/{preprocessed,spikesorted,postprocessed,curated,visualization}/...`) and emits the AIND-data-schema metadata files (`processing.json`, `data_description.json`).

It hard-asserts that exactly one `ecephys_*` folder is present in `data/`, so we synthesize a minimal one (`ecephys_toy/subject.json` + `data_description.json`).

## Imports and deps

In [1]:
import json
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "aind-data-schema", "aind-metadata-upgrader",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv


Resolved 35 packages in 4.01s


   Building pymongo==4.3.3


      Built pymongo==4.3.3
 Downloaded botocore
Prepared 13 packages in 1.99s
Uninstalled 2 packages in 11ms
Installed 15 packages in 59ms
 + aind-data-access-api==1.10.0
 + aind-metadata-upgrader==0.13.8
 + bcrypt==5.0.0
 + boto3==1.42.97
 + botocore==1.42.97
 + dnspython==2.8.0
 + invoke==3.0.3
 + jmespath==1.1.0
 + paramiko==4.0.0
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2
 + pymongo==4.3.3
 + pynacl==1.6.2
 + s3transfer==0.16.1
 + sshtunnel==0.4.0


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'aind-data-schema', 'aind-metadata-upgrader'], returncode=0)

## Clone the capsule and seed `data/`

We aggregate everything the previous notebooks produced into one flat data folder, plus a minimal `ecephys_toy/` session folder so the collector's `assert len(ecephys_sessions) == 1` is satisfied.

In [3]:
rc_repo = Path("/tmp/aind-ephys-results-collector")
if not rc_repo.exists():
    subprocess.run(
        [
            "git", "clone", "--depth=1",
            "https://github.com/AllenNeuralDynamics/aind-ephys-results-collector.git",
            str(rc_repo),
        ],
        check=True,
    )

data_dir = rc_repo / "data"
results_dir = rc_repo / "results"
data_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

for stale in list(data_dir.iterdir()) + list(results_dir.iterdir()):
    shutil.rmtree(stale) if stale.is_dir() else stale.unlink()

preproc_results = Path("/tmp/aind-ephys-preprocessing") / "results"
post_results = Path.cwd() / "postprocessing_results"
cur_results = Path.cwd() / "curation_results"
viz_results = Path.cwd() / "visualization_results"
sort_results = Path.cwd() / "spikesort_results"
dispatch_results = Path.cwd() / "dispatch_results"

for src in (preproc_results, sort_results, post_results, cur_results, viz_results, dispatch_results):
    assert src.exists(), f"{src} missing - run earlier notebooks first."
    for entry in src.iterdir():
        dest = data_dir / entry.name
        if dest.exists():
            continue
        shutil.copytree(entry, dest) if entry.is_dir() else shutil.copy2(entry, dest)

# Synthesize a minimal ecephys session folder so the collector's session
# detection (`ecephys_*` folder in data/) succeeds.
session_folder = data_dir / "ecephys_toy"
session_folder.mkdir()
(session_folder / "subject.json").write_text(json.dumps({"subject_id": "000000"}))
(session_folder / "data_description.json").write_text(json.dumps({
    "name": "ecephys_toy",
    "schema_version": "2.0.0",
    "creation_time": datetime.now(timezone.utc).isoformat(),
    "institution": {"name": "Allen Institute for Neural Dynamics", "abbreviation": "AIND"},
    "modalities": [{"name": "Extracellular electrophysiology", "abbreviation": "ecephys"}],
    "subject_id": "000000",
    "investigators": [{"name": "unknown"}],
    "funding_source": [{"funder": {"name": "Allen Institute for Neural Dynamics", "abbreviation": "AIND"}}],
    "project_name": "toy_pipeline",
    "data_level": "raw",
    "tags": [],
}, indent=2))

print("Seeded data dir:")
for p in sorted(data_dir.iterdir()):
    print(" ", p.name)

Cloning into '/tmp/aind-ephys-results-collector'...


Seeded data dir:
  binary_block0_None_recording1.json
  data_process_curation_block0_None_recording1.json
  data_process_postprocessing_block0_None_recording1.json
  data_process_preprocessing_block0_None_recording1.json
  data_process_spikesorting_block0_None_recording1.json
  data_process_visualization_block0_None_recording1.json
  ecephys_toy
  job_0.json
  postprocessed_block0_None_recording1.zarr
  preprocessed_block0_None_recording1
  preprocessed_block0_None_recording1.json
  preprocessedviz_block0_None_recording1.json
  qc_block0_None_recording1.npy
  spikesorted_block0_None_recording1
  unit_classifier_block0_None_recording1.csv
  visualization_block0_None_recording1


## Run the collector

The collector has no `params.json` — only CLI flags. We pass `--process-name sorted` (default) and skip `--pipeline-data-path` (used for symlink-resolution in HPC mode).

In [4]:
argv = [
    "python", "-u", "run_capsule.py",
    "--process-name", "sorted",
]
print("Running:", " ".join(argv))
result = subprocess.run(
    argv,
    cwd=rc_repo / "code",
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)
    raise RuntimeError(f"results-collector run failed with code {result.returncode}")

Running: python -u run_capsule.py --process-name sorted




COLLECTING RESULTS
Loaded session name from JSON files: session0
Copying preprocessed folders to results:
	block0_None_recording1
Copying spikesorted folders to results:
	block0_None_recording1
Copying postprocessed and curated folders to results:
	block0_None_recording1
		Remapping recording path for postprocessed to ../results/postprocessed/block0_None_recording1.zarr
	Resolving path for folder_path - ../../../../../Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/toy_example_recording
Copying visualization outputs to results:
Generating processing metadata
Generating data_description metadata
Failed to instantiate data description: 2 validation errors for DataDescription
funding_source.0.funder
  Input tag 'Allen Institute for Neural Dynamics' found using 'name' does not match any of the expected tags: 'Allen Institute', 'Chan Zuckerberg Initiative', 'MBF Bioscience', "Michael J. Fox Foundation for Parkinson's Research", 'National Center for Complementary a

## Copy outputs next to the notebook

In [5]:
local_results_dir = Path.cwd() / "collected_results"
if local_results_dir.exists():
    shutil.rmtree(local_results_dir)
shutil.copytree(results_dir, local_results_dir)

for p in sorted(local_results_dir.iterdir()):
    rel = p.relative_to(local_results_dir)
    kind = "dir" if p.is_dir() else f"{p.stat().st_size} bytes"
    print(f"  {rel}  ({kind})")

  curated  (dir)
  data_description.json  (1425 bytes)
  postprocessed  (dir)
  preprocessed  (dir)
  processing.json  (16283 bytes)
  spikesorted  (dir)
  subject.json  (24 bytes)
  visualization  (dir)
  visualization_output.json  (2 bytes)


## Inspect the collected processing.json

In [6]:
processing = json.loads((local_results_dir / "processing.json").read_text())
print("schema_version:", processing.get("schema_version"))
for pipeline in processing.get("pipelines", []):
    print("pipeline:", pipeline.get("name"))
process_names = [d.get("name") for d in processing.get("data_processes", [])]
print("data_processes:", process_names)

schema_version: 2.2.5
pipeline: AIND Ephys Pipeline
data_processes: ['Ephys preprocessing - block0_None_recording1', 'Spike sorting - block0_None_recording1', 'Ephys postprocessing - block0_None_recording1', 'Ephys curation - block0_None_recording1', 'Ephys visualization - block0_None_recording1']
